In [1]:
import os
import pandas as pd
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from skimage.transform import resize

In [2]:
DATA_ROOT = "data/heartbeat/"
IMG_SIZE = (128, 128)
SR = 22050
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

In [3]:
df_a = pd.read_csv(os.path.join(DATA_ROOT, "set_a.csv"))
df_b = pd.read_csv(os.path.join(DATA_ROOT, "set_b.csv"))

df = pd.concat([df_a, df_b], ignore_index=True)

# Keep only labeled samples
df = df[df["label"].notna()]

# Binary mapping: normal = 0, abnormal = 1
df["binary_label"] = df["label"].apply(lambda x: 0 if x == "normal" else 1)

# Create full file paths
def build_path(row):
    folder = "set_a" if row["dataset"] == "a" else "set_b"
    return os.path.join(DATA_ROOT, folder, row["fname"])

df["path"] = df.apply(build_path, axis=1)

file_paths = df["path"].values
labels = df["binary_label"].values

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    file_paths, labels, test_size=0.3, stratify=labels, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [5]:
def wav_to_mel(path):
    y, sr = librosa.load(path.decode("utf-8"), sr=SR)
    
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=128, n_fft=1024, hop_length=512
    )
    
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # Normalize 0–1
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    
    mel_resized = resize(mel_db, IMG_SIZE, mode="reflect", anti_aliasing=True)
    
    return mel_resized.astype(np.float32)

def load_sample(path, label):
    mel = tf.py_function(wav_to_mel, [path], tf.float32)
    mel.set_shape((128, 128))
    mel = tf.expand_dims(mel, -1)
    return mel, label

In [6]:
def create_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths))
    ds = ds.map(load_sample, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(X_train, y_train)
val_ds   = create_dataset(X_val, y_val, shuffle=False)
test_ds  = create_dataset(X_test, y_test, shuffle=False)

In [7]:
def build_model():
    inputs = tf.keras.layers.Input(shape=(128,128,1))

    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)

    x = tf.keras.layers.Conv2D(96, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)

model = build_model()
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 1)]     0         
                                                                 
 conv2d (Conv2D)             (None, 128, 128, 32)      320       
                                                                 
 batch_normalization (BatchN  (None, 128, 128, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 64, 64, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 64)        18496     
                                                                 
 batch_normalization_1 (Batc  (None, 64, 64, 64)       256   

In [ ]:
logdir = "logs/heartbeat/run1"
tb_cb = tf.keras.callbacks.TensorBoard(logdir)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[
        tb_cb,
        tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
    ]
)


Epoch 1/40


In [ ]:
y_true, y_pred = [], []

for x, y in test_ds:
    preds = model.predict(x).ravel()
    y_pred.extend((preds > 0.5).astype(int))
    y_true.extend(y.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))